# LEGO Sorter - Quick Experiments

Lightweight notebook for quick Blender experiments via MCP.

**Use this for**: Quick tests, scene inspection, parameter experimentation

**Use full pipeline notebook for**: Complete simulation runs

In [ ]:
# Quick Setup
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), "utils"))
from utils.blender_mcp_client import BlenderMCPClient

client = BlenderMCPClient(timeout=120)
print("✅ Ready" if client.test_connection() else "❌ Connection failed")

## Quick Actions

In [ ]:
# Get Scene Info
client.execute_code("""
import bpy
print(f"Objects: {len(bpy.data.objects)}")
print(f"Collections: {len(bpy.data.collections)}")
for col in bpy.data.collections:
    print(f"  - {col.name}: {len(col.objects)} objects")
""", "Scene Info")

In [ ]:
# Clear Scene
client.execute_script_file('blender/clear_scene.py', 'Clear')

In [ ]:
# Create Test Bucket
client.execute_script_file('blender/create_sorting_bucket.py', 'Bucket')

## Experimentation Area

Use this cell for quick Blender code experiments:

In [ ]:
# Your experiment code here
code = """
import bpy

# Example: List all mesh objects
meshes = [obj for obj in bpy.data.objects if obj.type == 'MESH']
print(f"Found {len(meshes)} mesh objects:")
for obj in meshes[:10]:  # Show first 10
    print(f"  - {obj.name}: {len(obj.data.vertices)} vertices")
"""

client.execute_code(code, "Experiment")

## Parameter Testing

Test different parameter values quickly:

In [ ]:
# Test different bucket sizes
sizes = [0.20, 0.24, 0.28]  # Different top sizes

for size in sizes:
    print(f"\n=== Testing bucket size: {size} ===")
    
    code = f"""
import bpy

# Clear previous
col = bpy.data.collections.get("test_bucket")
if col:
    for obj in col.objects:
        bpy.data.objects.remove(obj, do_unlink=True)
    bpy.data.collections.remove(col)

# Create test bucket
col = bpy.data.collections.new("test_bucket")
bpy.context.scene.collection.children.link(col)

bpy.ops.mesh.primitive_cube_add(size={size}, location=(0, 0, 0))
obj = bpy.context.active_object
if obj:
    obj.name = f"TestBucket_{{size}}"
    col.objects.link(obj)
    print(f"Created bucket with size {{size}}")
"""
    
    client.execute_code(code, f"Test Size {size}")

## Physics Debugging

In [ ]:
# Check physics setup
client.execute_code("""
import bpy

rigid_bodies = [obj for obj in bpy.data.objects if obj.rigid_body]
print(f"Rigid body objects: {len(rigid_bodies)}")

for obj in rigid_bodies[:5]:  # Show first 5
    rb = obj.rigid_body
    print(f"\n{obj.name}:")
    print(f"  Type: {rb.type}")
    print(f"  Mass: {rb.mass}")
    print(f"  Friction: {rb.friction}")
""", "Physics Check")

In [ ]:
# Check current frame state
client.execute_code("""
import bpy

scene = bpy.context.scene
print(f"Current frame: {scene.frame_current}")
print(f"Frame range: {scene.frame_start} - {scene.frame_end}")

# Sample object positions
for obj in list(bpy.data.objects)[:3]:
    loc = obj.location
    print(f"{obj.name}: ({loc.x:.3f}, {loc.y:.3f}, {loc.z:.3f})")
""", "Frame State")